# ✍ BrandMind AI — Week 4
## Creative Content Hub — Tagline & Slogan Generation
**Tools:** Gemini API · HuggingFace Transformers · NLTK

In [ ]:
!pip install google-generativeai nltk transformers pandas -q

In [ ]:
import nltk, re, json, pandas as pd
nltk.download(['punkt','stopwords','punkt_tab'], quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import google.generativeai as genai
print('Ready')

## 1. Text Preprocessing — Slogan Dataset

In [ ]:
# Load slogan dataset
try:
    df_slogans = pd.read_csv('/content/drive/MyDrive/BrandMind_Datasets/slogans.csv')
except:
    df_slogans = pd.DataFrame({'slogan': [
        'Innovation for a better tomorrow',
        'Quality you can trust, every time',
        'Building the future, together',
        'Excellence in every detail',
        'Your success is our mission',
    ] * 100, 'industry': ['Technology','Finance','Healthcare','Retail','Education'] * 100})

stop_words = set(stopwords.words('english'))

def preprocess_slogan(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)       # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Normalise spaces
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

df_slogans['tokens'] = df_slogans['slogan'].apply(preprocess_slogan)
df_slogans['clean']  = df_slogans['tokens'].apply(lambda t: ' '.join(t))
df_slogans['word_count'] = df_slogans['slogan'].apply(lambda x: len(x.split()))

print(df_slogans[['slogan','clean','word_count']].head())

## 2. Slogan Generation via Gemini API

In [ ]:
# Configure Gemini
GEMINI_API_KEY = 'YOUR_API_KEY_HERE'  # Replace with your key
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-pro')

def generate_slogans(company, industry, tone, audience, n=5):
    prompt = f"""You are a world-class brand copywriter.
Company: {company}
Industry: {industry}
Brand Tone: {tone}
Target Audience: {audience}

Generate {n} unique, punchy, memorable slogans for this brand.
Each should be max 8 words and capture the brand essence.
Return as JSON: {{"slogans": [{{"text": "...", "vibe": "one-word description"}}]}}"""
    try:
        response = model.generate_content(prompt)
        data = json.loads(response.text.replace('```json','').replace('```','').strip())
        return data['slogans']
    except Exception as e:
        print(f'API error: {e} — using fallback')
        return [
            {'text': f'{company} — Built for what\'s next.', 'vibe': 'Forward-looking'},
            {'text': f'Think different. Choose {company}.', 'vibe': 'Bold'},
            {'text': f'{company}. Simply brilliant.', 'vibe': 'Minimalist'},
            {'text': f'Where {industry} meets excellence.', 'vibe': 'Premium'},
            {'text': f'The future of {industry} starts here.', 'vibe': 'Inspirational'},
        ]

slogans = generate_slogans('BrandMind', 'Technology', 'Innovative', 'Startup founders')
for s in slogans:
    print(f"  [{s['vibe']}] {s['text']}")

## 3. Preliminary Multilingual Translation (W4 requirement)

In [ ]:
def translate_slogan(slogan, target_lang):
    prompt = f"""Translate this brand slogan to {target_lang}. Preserve tone and punchiness.
Slogan: '{slogan}'
Return ONLY the translated text."""
    try:
        return model.generate_content(prompt).text.strip()
    except:
        return f'[{target_lang} translation of: {slogan}]'

# Preview multilingual (preliminary — full version in Week 8)
selected_slogan = slogans[0]['text']
print(f'Original: {selected_slogan}\n')
for lang in ['Spanish', 'French', 'German']:
    translated = translate_slogan(selected_slogan, lang)
    print(f'{lang}: {translated}')

In [ ]:
# Save slogans to file (downloadable)
import os
os.makedirs('outputs', exist_ok=True)
with open('outputs/slogans.json', 'w') as f:
    json.dump({'company': 'BrandMind', 'slogans': slogans}, f, indent=2)
print('Saved: outputs/slogans.json')

slogan_txt = '\n'.join([f"{i+1}. [{s['vibe']}] {s['text']}" for i, s in enumerate(slogans)])
with open('outputs/slogans.txt', 'w') as f:
    f.write(slogan_txt)
print('Saved: outputs/slogans.txt')

## Week 4 Complete ✅
- ✅ NLTK preprocessing (tokenise, lowercase, remove punctuation)
- ✅ Gemini API slogan generation with persona input
- ✅ 5 slogans per company
- ✅ Preliminary multilingual translation
- ✅ Downloadable slogan list (JSON + TXT)